# 14 RAGChecker Evaluation

Το notebook προετοιμάζει τα QA και retrieval outputs στη μορφή που απαιτεί το RAGChecker, εκτελεί την αξιολόγηση και αποθηκεύει τα συνοπτικά και αναλυτικά αποτελέσματα.


## 1. Εγκατάσταση

Το κελί εγκαθιστά τις πρόσθετες βιβλιοθήκες όταν δεν υπάρχουν ήδη στο περιβάλλον εκτέλεσης. Σε Colab, Kaggle ή Jupyter ενδέχεται να απαιτηθεί επανεκκίνηση του kernel μετά την εγκατάσταση.


In [ ]:
# %pip install -q ragchecker spacy pandas numpy
# !python -m spacy download en_core_web_sm

print("Uncomment the installation lines above and run them once if needed.")


## 2. Εισαγωγές βιβλιοθηκών


In [ ]:

import os
import json
import ast
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd

try:
    from ragchecker import RAGResults, RAGChecker
    from ragchecker.metrics import all_metrics
    RAGCHECKER_AVAILABLE = True
    RAGCHECKER_IMPORT_ERROR = None
except Exception as exc:
    RAGResults = None
    RAGChecker = None
    all_metrics = None
    RAGCHECKER_AVAILABLE = False
    RAGCHECKER_IMPORT_ERROR = f"{type(exc).__name__}: {exc}"
    print("RAGChecker import failed; placeholder outputs will be written.")
    print(RAGCHECKER_IMPORT_ERROR)


## 3. Ρυθμίσεις

Το notebook χρησιμοποιεί τα αρχεία QA και retrieval που παράγονται στα προηγούμενα στάδια. Το API key διαβάζεται μόνο από μεταβλητή περιβάλλοντος και δεν αποθηκεύεται στον κώδικα.


In [ ]:

# -----------------------------
# ΡΥΘΜΙΣΕΙΣ API KEY / MODEL
# -----------------------------
EXTRACTOR_NAME = "gpt-4o-mini"
CHECKER_NAME = "gpt-4o-mini"

BATCH_SIZE_EXTRACTOR = 4
BATCH_SIZE_CHECKER = 4

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    BASE_DIR = CURRENT_DIR.parent
else:
    BASE_DIR = CURRENT_DIR

DATA_DIR = BASE_DIR / "data"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
QA_DIR = PROCESSED_DIR / "qa_results"
RETRIEVAL_DIR = PROCESSED_DIR / "retrieval_results"
EVAL_DIR = PROCESSED_DIR / "evaluation"

EVAL_DIR.mkdir(parents=True, exist_ok=True)

WORKING_DATASET_CSV_PATH = INTERIM_DIR / "financebench_open_source_working.csv"

QA_PATHS = {
    "dense": QA_DIR / "rag_qa_results_dense.csv",
    "hybrid": QA_DIR / "rag_qa_results_hybrid.csv",
    "reranked": QA_DIR / "rag_qa_results_hybrid_reranked.csv",
}

RETRIEVAL_PATHS = {
    "dense": RETRIEVAL_DIR / "retrieval_results_dense.csv",
    "hybrid": RETRIEVAL_DIR / "retrieval_results_hybrid.csv",
    "reranked": RETRIEVAL_DIR / "retrieval_results_hybrid_reranked.csv",
}

RUN_NAMES = ["dense", "hybrid", "reranked"]
RUN_NAME = "dense"
INPUT_TYPE = "csv"

CSV_MAPPING = {
    "query_id": "financebench_id",
    "query": "question",
    "gt_answer": "expected_answer",
    "response": "generated_answer",
}
RETRIEVED_CONTEXT_COLUMNS = ["context_text"]

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
RUN_RAGCHECKER = bool(OPENAI_API_KEY) and RAGCHECKER_AVAILABLE

if not OPENAI_API_KEY:
    print("OPENAI_API_KEY not found; placeholder RAGChecker outputs will be written.")
if not RAGCHECKER_AVAILABLE:
    print("RAGChecker unavailable; placeholder RAGChecker outputs will be written.")

print("BASE_DIR:", BASE_DIR)
print("EVAL_DIR:", EVAL_DIR)
for run_name in RUN_NAMES:
    print(f"{run_name} QA exists:", QA_PATHS[run_name].exists(), "->", QA_PATHS[run_name])
    print(f"{run_name} retrieval exists:", RETRIEVAL_PATHS[run_name].exists(), "->", RETRIEVAL_PATHS[run_name])
print("RUN_RAGCHECKER:", RUN_RAGCHECKER)


In [ ]:
if not RUN_RAGCHECKER:
    reason = RAGCHECKER_IMPORT_ERROR or "OPENAI_API_KEY not available"
    raise RuntimeError(f"RAGChecker is required for canonical evaluation: {reason}")


## 4. Βοηθητικές συναρτήσεις μετατροπής schema

Το RAGChecker θέλει αυτό το format:

```json
{
  "results": [
    {
      "query_id": "1",
      "query": "...",
      "gt_answer": "...",
      "response": "...",
      "retrieved_context": [
        {"doc_id": "doc_1", "text": "..."},
        {"doc_id": "doc_2", "text": "..."}
      ]
    }
  ]
}
```


In [ ]:
def safe_isna(x: Any) -> bool:
    try:
        return pd.isna(x)
    except Exception:
        return False

def normalize_text(x: Any) -> str:
    if x is None or safe_isna(x):
        return ""
    return str(x).strip()

def parse_retrieved_context(value: Any) -> List[Dict[str, str]]:
    """
    Μετατρέπει διάφορες πιθανές μορφές retrieved context
    σε λίστα από {"doc_id": ..., "text": ...}.
    """
    if value is None or safe_isna(value):
        return []

    # Ήδη λίστα
    if isinstance(value, list):
        out = []
        for i, item in enumerate(value):
            if isinstance(item, dict):
                text = normalize_text(item.get("text", ""))
                doc_id = normalize_text(item.get("doc_id", f"doc_{i+1}"))
                if text:
                    out.append({"doc_id": doc_id or f"doc_{i+1}", "text": text})
            else:
                text = normalize_text(item)
                if text:
                    out.append({"doc_id": f"doc_{i+1}", "text": text})
        return out

    # Αν είναι dict
    if isinstance(value, dict):
        text = normalize_text(value.get("text", ""))
        if text:
            return [{"doc_id": normalize_text(value.get("doc_id", "doc_1")) or "doc_1", "text": text}]
        return []

    # String που μπορεί να είναι JSON / python literal / plain text
    if isinstance(value, str):
        text_value = value.strip()
        if not text_value:
            return []

        # Προσπάθεια JSON
        try:
            parsed = json.loads(text_value)
            return parse_retrieved_context(parsed)
        except Exception:
            pass

        # Προσπάθεια ast.literal_eval
        try:
            parsed = ast.literal_eval(text_value)
            return parse_retrieved_context(parsed)
        except Exception:
            pass

        # Εναλλακτικά: απλό κείμενο chunk
        return [{"doc_id": "doc_1", "text": text_value}]

    # Εναλλακτική περίπτωση
    text = normalize_text(value)
    return [{"doc_id": "doc_1", "text": text}] if text else []


def build_context_from_multiple_columns(row: pd.Series, context_cols: List[str]) -> List[Dict[str, str]]:
    chunks = []
    for i, col in enumerate(context_cols, start=1):
        if col in row.index:
            text = normalize_text(row[col])
            if text:
                chunks.append({"doc_id": f"{col}_{i}", "text": text})
    return chunks


def df_to_ragchecker_records(
    df: pd.DataFrame,
    mapping: Dict[str, str],
    context_cols: Optional[List[str]] = None
) -> List[Dict[str, Any]]:
    context_cols = context_cols or []
    records = []

    required_base = ["query_id", "query", "gt_answer", "response"]
    for field in required_base:
        if mapping.get(field) not in df.columns:
            raise ValueError(f"Missing required column for '{field}': {mapping.get(field)!r}")

    for idx, row in df.iterrows():
        if context_cols:
            retrieved_context = build_context_from_multiple_columns(row, context_cols)
        else:
            rc_col = mapping.get("retrieved_context")
            if rc_col not in df.columns:
                raise ValueError(f"Missing retrieved_context column: {rc_col!r}")
            retrieved_context = parse_retrieved_context(row[rc_col])

        record = {
            "query_id": normalize_text(row[mapping["query_id"]]) or str(idx),
            "query": normalize_text(row[mapping["query"]]),
            "gt_answer": normalize_text(row[mapping["gt_answer"]]),
            "response": normalize_text(row[mapping["response"]]),
            "retrieved_context": retrieved_context,
        }
        records.append(record)
    return records


def load_input_as_ragchecker_json(
    input_path: Path,
    input_type: str,
    mapping: Optional[Dict[str, str]] = None,
    context_cols: Optional[List[str]] = None,
) -> Dict[str, Any]:
    input_type = input_type.lower().strip()

    if input_type == "csv":
        df = pd.read_csv(input_path)
        print(f"Loaded CSV shape: {df.shape}")
        display(df.head(3))
        records = df_to_ragchecker_records(df, mapping=mapping or {}, context_cols=context_cols or [])
        return {"results": records}

    if input_type == "json":
        with open(input_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        # Αν είναι ήδη στο σωστό format
        if isinstance(data, dict) and "results" in data:
            return data

        # Αν είναι λίστα από records
        if isinstance(data, list):
            return {"results": data}

        raise ValueError("Unsupported JSON structure. Expected dict with 'results' or list of records.")

    raise ValueError("INPUT_TYPE must be either 'csv' or 'json'.")


In [ ]:
CSV_MAPPING = {
    "query_id": "financebench_id",
    "query": "question",
    "gt_answer": "expected_answer",
    "response": "generated_answer",
}

RETRIEVED_CONTEXT_COLUMNS = ["context_text"]

print("CSV_MAPPING:", CSV_MAPPING)
print("RETRIEVED_CONTEXT_COLUMNS:", RETRIEVED_CONTEXT_COLUMNS)


## 5. Φόρτωση και μετατροπή δεδομένων


In [ ]:

RAGCHECKER_SUMMARY_METRICS = {
    "overall_metrics": {
        "precision": 0.0,
        "recall": 0.0,
        "f1": 0.0,
    },
    "retriever_metrics": {
        "claim_recall": 0.0,
        "context_precision": 0.0,
    },
    "generator_metrics": {
        "context_utilization": 0.0,
        "faithfulness": 0.0,
        "hallucination": 0.0,
        "noise_sensitivity_in_relevant": 0.0,
        "noise_sensitivity_in_irrelevant": 0.0,
        "self_knowledge": 0.0,
    },
}


def make_placeholder_results(ragchecker_input: Dict[str, Any], reason: str) -> Dict[str, Any]:
    rows = []
    for item in ragchecker_input.get("results", []):
        row = dict(item)
        row["metrics"] = {
            "f1": 0.0,
            "faithfulness": 0.0,
            "hallucination": 0.0,
        }
        row["ragchecker_status"] = "skipped"
        row["ragchecker_skip_reason"] = reason
        rows.append(row)
    return {
        "metrics": RAGCHECKER_SUMMARY_METRICS,
        "results": rows,
        "ragchecker_status": "skipped",
        "ragchecker_skip_reason": reason,
    }


def extract_summary_rows(results_data: Dict[str, Any]) -> pd.DataFrame:
    summary_rows = []
    metrics_root = results_data.get("metrics", {})
    if not isinstance(metrics_root, dict):
        metrics_root = {}

    for section in ["overall_metrics", "retriever_metrics", "generator_metrics"]:
        section_metrics = metrics_root.get(section, {})
        if isinstance(section_metrics, dict):
            for metric_name, metric_value in section_metrics.items():
                summary_rows.append({
                    "metric_group": section,
                    "metric_name": metric_name,
                    "metric_value": float(metric_value) if metric_value is not None else 0.0,
                })

    if not summary_rows:
        for section, section_metrics in RAGCHECKER_SUMMARY_METRICS.items():
            for metric_name, metric_value in section_metrics.items():
                summary_rows.append({
                    "metric_group": section,
                    "metric_name": metric_name,
                    "metric_value": metric_value,
                })

    return pd.DataFrame(summary_rows).sort_values(
        ["metric_group", "metric_name"]
    ).reset_index(drop=True)


def save_ragchecker_details(results_data: Dict[str, Any], path: Path) -> pd.DataFrame:
    detail_items = results_data.get("results", [])
    details_df = pd.json_normalize(detail_items)

    for col in ["metrics.f1", "metrics.faithfulness", "metrics.hallucination"]:
        if col not in details_df.columns:
            details_df[col] = 0.0
        details_df[col] = pd.to_numeric(details_df[col], errors="coerce").fillna(0.0)

    details_df.to_csv(path, index=False, encoding="utf-8-sig")
    return details_df


ragchecker_outputs = {}

for run_name in RUN_NAMES:
    input_path = QA_PATHS[run_name]
    transformed_json_path = EVAL_DIR / f"ragchecker_input_{run_name}.json"
    results_json_path = EVAL_DIR / f"ragchecker_results_{run_name}.json"
    summary_csv_path = EVAL_DIR / f"ragchecker_summary_{run_name}.csv"
    details_csv_path = EVAL_DIR / f"ragchecker_details_{run_name}.csv"

    if not input_path.exists():
        raise FileNotFoundError(f"Missing QA input for RAGChecker run '{run_name}': {input_path}")

    print(f"\n=== RAGChecker run: {run_name} ===")
    ragchecker_input = load_input_as_ragchecker_json(
        input_path=input_path,
        input_type=INPUT_TYPE,
        mapping=CSV_MAPPING,
        context_cols=RETRIEVED_CONTEXT_COLUMNS,
    )

    with open(transformed_json_path, "w", encoding="utf-8") as f:
        json.dump(ragchecker_input, f, ensure_ascii=False, indent=2)
    print(f"Saved transformed input to: {transformed_json_path}")

    if RUN_RAGCHECKER:
        try:
            rag_results = RAGResults.from_json(json.dumps(ragchecker_input, ensure_ascii=False))
            evaluator = RAGChecker(
                extractor_name=EXTRACTOR_NAME,
                checker_name=CHECKER_NAME,
                batch_size_extractor=BATCH_SIZE_EXTRACTOR,
                batch_size_checker=BATCH_SIZE_CHECKER,
            )
            evaluator.evaluate(rag_results, all_metrics)
            results_data = json.loads(rag_results.to_json())
            results_data["ragchecker_status"] = "completed"
            results_data["ragchecker_skip_reason"] = ""
            print("Evaluation completed.")
        except Exception as exc:
            reason = f"{type(exc).__name__}: {exc}"
            print(f"RAGChecker failed for {run_name}; writing placeholder outputs.")
            print(reason)
            results_data = make_placeholder_results(ragchecker_input, reason)
    else:
        reason = RAGCHECKER_IMPORT_ERROR or "OPENAI_API_KEY not available"
        results_data = make_placeholder_results(ragchecker_input, reason)

    with open(results_json_path, "w", encoding="utf-8") as f:
        json.dump(results_data, f, ensure_ascii=False, indent=2)
    print(f"Saved raw results to: {results_json_path}")

    summary_df = extract_summary_rows(results_data)
    summary_df.to_csv(summary_csv_path, index=False, encoding="utf-8-sig")
    print(f"Saved summary to: {summary_csv_path}")

    details_df = save_ragchecker_details(results_data, details_csv_path)
    print(f"Saved details to: {details_csv_path}")

    ragchecker_outputs[run_name] = {
        "input": ragchecker_input,
        "results": results_data,
        "summary": summary_df,
        "details": details_df,
        "paths": {
            "transformed_json": transformed_json_path,
            "results_json": results_json_path,
            "summary_csv": summary_csv_path,
            "details_csv": details_csv_path,
        }
    }

RUN_NAME = "dense"
ragchecker_input = ragchecker_outputs[RUN_NAME]["input"]
results_data = ragchecker_outputs[RUN_NAME]["results"]
summary_df = ragchecker_outputs[RUN_NAME]["summary"]
details_df = ragchecker_outputs[RUN_NAME]["details"]

print("\nRAGChecker outputs written for:", list(ragchecker_outputs))
display(summary_df)
display(details_df.head())


In [ ]:
incomplete_runs = [
    run_name
    for run_name, output in ragchecker_outputs.items()
    if output["results"].get("ragchecker_status") != "completed"
]
if incomplete_runs:
    raise RuntimeError(f"RAGChecker evaluation incomplete for runs: {incomplete_runs}")


## 6. Γρήγορο validation του transformed input


In [ ]:
print("Handled by the consolidated multi-run RAGChecker cell above.")


## 7. Εκτέλεση RAGChecker

Το RAGChecker εκτελεί claim extraction και claim checking με LLM. Η εκτέλεση απαιτεί σωστά ορισμένο API key και provider configuration.

Τα διαθέσιμα metric groups είναι:
- `overall_metrics`
- `retriever_metrics`
- `generator_metrics`
- `all_metrics`


In [ ]:
print("Handled by the consolidated multi-run RAGChecker cell above.")


In [ ]:
print("Handled by the consolidated multi-run RAGChecker cell above.")


## 8. Αποθήκευση raw αποτελεσμάτων


In [ ]:
print("Handled by the consolidated multi-run RAGChecker cell above.")


## 9. Φόρτωση αποτελεσμάτων και σύνοψη metrics


In [ ]:
print("Handled by the consolidated multi-run RAGChecker cell above.")


In [ ]:
print("Handled by the consolidated multi-run RAGChecker cell above.")


In [ ]:
print("Handled by the consolidated multi-run RAGChecker cell above.")


## 10. Αναλυτικά per-example αποτελέσματα

Το ακριβές schema μπορεί να διαφέρει λίγο ανά έκδοση.  
Το παρακάτω cell προσπαθεί να βγάλει όσο πιο χρήσιμο table γίνεται.


In [ ]:
print("Handled by the consolidated multi-run RAGChecker cell above.")


## 11. Ανάλυση αποτυχιών

Αυτό βοηθάει να βρεις δύσκολα queries, μικρό context, ή περιπτώσεις όπου λείπουν retrieved chunks.


In [ ]:
print("Handled by the consolidated multi-run RAGChecker cell above.")


## 12. Προαιρετικό: σύγκριση πολλών runs

Η ενότητα συγκρίνει πολλαπλά `ragchecker_results.json` από dense, hybrid και reranked runs.


In [ ]:
print("Handled by the consolidated multi-run RAGChecker cell above.")


## 13. Σημειώσεις χρήσης

Για την αξιολόγηση χρησιμοποιούνται, ανά ερώτηση, τα πεδία `question`, `gold answer`, `generated answer` και τα top-k retrieved chunks. Το notebook μετατρέπει αυτά τα δεδομένα στη μορφή του RAGChecker, εκτελεί την αξιολόγηση και αποθηκεύει summary και detailed outputs για σύγκριση μεταξύ runs.
